In [1]:
import torch
import random
import numpy as np
import pandas as pd
import re
import emoji
import json
from tqdm import tqdm
import PyPDF2
import seaborn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.metrics import confusion_matrix, classification_report, f1_score
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForCausalLM
)

from sentence_transformers import SentenceTransformer, util

SEED = 42
TAM_MODEL = "Qwen/Qwen2.5-3B-Instruct"

MAX_LEN = 128
BATCH_SIZE = 8
EPOCHS = 2
LR = 2e-5

TAM_LABELS = ["PE", "EE", "SI", "FC"]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


d:\PPTI 15 ARTEMIS\Semester 8 (Skripsi)\TAM UTAUT\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


In [2]:
import re
import emoji

slang_dict = {
    "ga": "tidak",
    "gak": "tidak",
    "gk": "tidak",
    "nggak": "tidak",
    "tp": "tapi",
    "jd": "jadi",
    "bgt": "banget",
    "yg": "yang",
    "krn": "karena"
}

def normalize_slang(text):
    words = text.split()
    return " ".join([slang_dict.get(w, w) for w in words])


def preprocess_text(text):
    text = str(text)

    # 1. Lowercase
    text = text.lower()

    # 2. Remove URL
    text = re.sub(r'http\S+|www\S+', ' ', text)

    # 3. Remove mention
    text = re.sub(r'@\w+', ' ', text)

    # 4. Remove hashtag symbol only (kata tetap)
    text = re.sub(r'#', '', text)

    # 5. Remove emoji
    text = emoji.replace_emoji(text, replace='')

    # 6. Slang normalization ringan
    text = normalize_slang(text)

    # 7. Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text


In [3]:
df_comment = pd.read_csv("datasets_final.csv", sep=";", encoding="utf-8-sig")
df_comment["comment"] = df_comment["comment"].astype(str).apply(preprocess_text)

In [4]:
df_rag=df_comment.sample(20,random_state=SEED)

df_rest=df_comment.drop(df_rag.index)

In [ ]:
from huggingface_hub import login
login("useyourtokenhere")

In [6]:
def load_pdf_text(pdf_path):
    text = ""
    with open(pdf_path, "rb") as file:
        reader = PyPDF2.PdfReader(file)
        for page in reader.pages:
            text += page.extract_text() + "\n"
    return text

utaut_text = load_pdf_text("utaut_docs.pdf")


In [7]:
def chunk_text(text, chunk_size=400, overlap=50):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = words[i:i + chunk_size]
        chunks.append(" ".join(chunk))
    return chunks

utaut_chunks = chunk_text(utaut_text)

embed_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

chunk_embeddings = embed_model.encode(
    utaut_chunks,
    convert_to_tensor=True
)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 566.80it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
def retrieve_context(query, top_k=3, threshold=0.4):
    query_embedding = embed_model.encode(query, convert_to_tensor=True)

    hits = util.semantic_search(query_embedding, chunk_embeddings, top_k=top_k)[0]

    filtered_chunks = []
    for hit in hits:
        if hit["score"] >= threshold:
            filtered_chunks.append(utaut_chunks[hit["corpus_id"]])

    if len(filtered_chunks) == 0:
        return "Gunakan definisi umum konstruk TAM-UTAUT."

    return "\n\n".join(filtered_chunks)


In [9]:
tam_tokenizer = AutoTokenizer.from_pretrained(TAM_MODEL)

tam_model = AutoModelForCausalLM.from_pretrained(
    TAM_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)

tam_model.eval()


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 434/434 [00:05<00:00, 79.44it/s, Materializing param=model.norm.weight]                               
Some parameters are on the meta device because they were offloaded to the cpu.


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-35): 36 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear(in_features=2048, out_features=256, bias=True)
          (v_proj): Linear(in_features=2048, out_features=256, bias=True)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((2048,), eps=1e-06)
    (ro

In [10]:
def build_tam_prompt(text):

    context = retrieve_context(text)

    return f"""
You are an expert in UTAUT (Unified Theory of Acceptance and Use of Technology).

Your task is to analyze a social media comment and assign scores to each UTAUT construct.

Use the theoretical reference below:

{context}

Definitions:
- PE (Performance Expectancy): perceived usefulness or benefits (e.g., improves productivity, helps tasks)
- EE (Effort Expectancy): ease of use (e.g., easy, difficult, confusing)
- SI (Social Influence): influence from others (e.g., trends, recommendations, viral)
- FC (Facilitating Conditions): supporting resources (e.g., system support, documentation, compatibility)

Scoring Rules:
- 0 = not mentioned at all
- 1–5 = mentioned with sentiment:
  1 = very negative
  2 = negative
  3 = neutral
  4 = positive
  5 = very positive

Important Rules:
- Assign >0 if the construct is explicitly OR implicitly indicated.
- Do not assign randomly.
- Be reasonable, not overly strict.

Output Rules:
- Return ONLY valid JSON
- No explanation
- No extra text

Comment:
{text}

Output format:
{{
  "PE": 0,
  "EE": 0,
  "SI": 0,
  "FC": 0
}}
"""

In [11]:
def safe_parse_json(result):

    try:
        json_text = re.search(r"\{.*\}", result, re.DOTALL).group()
        parsed = json.loads(json_text)

        clean = {}
        for k in TAM_LABELS:
            val = int(parsed.get(k, 0))
            val = max(0, min(5, val))
            clean[k] = val

        return clean

    except:
        return {k: 0 for k in TAM_LABELS}


In [12]:
def score_tam(text):

    prompt = build_tam_prompt(text)

    inputs = tam_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(tam_model.device)

    with torch.inference_mode():
        outputs = tam_model.generate(
            **inputs,
            max_new_tokens=120,
            temperature=0.0,
            do_sample=False
        )

    generated = outputs[0][inputs["input_ids"].shape[-1]:]
    result = tam_tokenizer.decode(generated, skip_special_tokens=True).strip()

    try:
        json_start = result.find("{")
        json_end = result.rfind("}") + 1
        json_text = result[json_start:json_end]

        parsed = json.loads(json_text)

        # pastikan selalu dict lengkap
        clean = {}
        for k in TAM_LABELS:
            clean[k] = int(parsed.get(k, 0))

        return clean

    except Exception as e:
        print("Parsing gagal:", result)
        return {k: 0 for k in TAM_LABELS}

In [13]:
tam_results = []

for text in tqdm(df_rag["comment"]):
    tam_results.append(score_tam(text))

tam_df = pd.DataFrame(tam_results)

df_rag = pd.concat(
    [df_rag.reset_index(drop=True),
     tam_df.reset_index(drop=True)],
    axis=1
)

100%|██████████| 20/20 [08:21<00:00, 25.09s/it]


In [ ]:
df_rag.to_csv("rag_labeled.csv", index=False)